# 02 — Model Training Walkthrough

Step-by-step walkthrough of training the BiLSTM+CNN threat detection model.
This notebook is for understanding — use `python train.py` for full training runs.

**Covers:**
- Loading preprocessed tensors
- Instantiating model variants
- Training loop with Focal Loss
- Loss/accuracy curve visualisation

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

with open('../configs/train_config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config loaded')

In [ ]:
# ── Check if processed data exists ─────────────────────────────────────
processed_dir = '../data/processed'
required = ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy']
missing = [f for f in required if not os.path.exists(os.path.join(processed_dir, f))]

if missing:
    print('[WARNING] Processed data not found. Run preprocessing first:')
    print('  python scripts/preprocess.py --input data/raw/ --output data/processed/')
    print(f'  Missing: {missing}')
else:
    X_train = np.load(os.path.join(processed_dir, 'X_train.npy'))
    y_train = np.load(os.path.join(processed_dir, 'y_train.npy'))
    X_val   = np.load(os.path.join(processed_dir, 'X_val.npy'))
    y_val   = np.load(os.path.join(processed_dir, 'y_val.npy'))
    print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
    print(f'X_val  : {X_val.shape}   | y_val  : {y_val.shape}')

In [ ]:
# ── Model Instantiation & Parameter Count ──────────────────────────────
from src.models.threat_model import ThreatDetectionModel
from src.models.lstm_encoder import BiLSTMEncoder
from src.models.cnn_classifier import CNNClassifier

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

variants = {
    'LSTM only': ThreatDetectionModel(hidden_dim=256, num_filters=0, num_classes=15),
    'CNN only':  ThreatDetectionModel(hidden_dim=0,   num_filters=128, num_classes=15),
    'Hybrid':    ThreatDetectionModel(hidden_dim=256, num_filters=128, num_classes=15),
}

print('Model variant parameter counts:')
for name, model in variants.items():
    print(f'  {name:12s}: {count_params(model):>10,} parameters')

In [ ]:
# ── Forward Pass Sanity Check ──────────────────────────────────────────
model = ThreatDetectionModel(num_classes=15).to(device)
model.eval()

dummy_input = torch.randn(4, 50, 78).to(device)
with torch.no_grad():
    logits = model(dummy_input)
    proba  = model.predict_proba(dummy_input)
    preds, confs = model.predict(dummy_input)

print(f'Input shape  : {dummy_input.shape}')
print(f'Logits shape : {logits.shape}')
print(f'Proba sums   : {proba.sum(dim=-1).tolist()}  (should be ~1.0)')
print(f'Predictions  : {preds.tolist()}')
print(f'Confidences  : {[f"{c:.3f}" for c in confs.tolist()]}')

In [ ]:
# ── Focal Loss Behaviour vs Cross-Entropy ──────────────────────────────
from src.models.focal_loss import FocalLoss
import torch.nn.functional as F

gammas = [0.0, 0.5, 1.0, 2.0, 5.0]
# Simulate: p_t from 0.01 (hard) to 0.99 (easy)
p_t_vals = np.linspace(0.01, 0.99, 100)

plt.figure(figsize=(8, 4))
for gamma in gammas:
    focal_weight = (1 - p_t_vals) ** gamma
    ce_loss = -np.log(p_t_vals)
    focal_loss = focal_weight * ce_loss
    plt.plot(p_t_vals, focal_loss, label=f'γ={gamma}')

plt.xlabel('p_t (probability of correct class)')
plt.ylabel('Loss value')
plt.title('Focal Loss vs Cross-Entropy (γ=0) for Different γ Values')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/focal_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: higher γ → more down-weighting of easy examples (high p_t)')

In [ ]:
# ── Load Training History (after training run) ─────────────────────────
history_path = '../results/logs/history.json'

if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#185FA5')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='#E24B4A')
    axes[0].set_title('Focal Loss — Training & Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['val_acc'], color='#1D9E75', label='Val Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylim([0, 1])
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../results/figures/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Best val_loss : {min(history["val_loss"]):.4f} (epoch {np.argmin(history["val_loss"])+1})')
    print(f'Best val_acc  : {max(history["val_acc"]):.4f} (epoch {np.argmax(history["val_acc"])+1})')
else:
    print('[INFO] No training history found yet.')
    print('Run: python train.py')
    print('Results will appear here after training completes.')